# SisFall Data Processing and SVM Training

This notebook processes the SisFall dataset, trains a Support Vector Machine (SVM) model, and exports the model as C code for embedded systems like STM32.

### Step 1: Install Required Libraries
First, we need to install the necessary Python libraries: `pandas`, `numpy`, `scikit-learn`, and `micromlgen`.

In [ ]:
!pip install pandas numpy scikit-learn micromlgen

  Preparing metadata (setup.py) ... done
  Created wheel for micromlgen: filename=micromlgen-1.1.28-py3-none-any.whl size=32152 sha256=f2ad6ab6910d95df5654c96565243d480760f07e5739fab8683380bb31b64ce9
  Stored in directory: /root/.cache/pip/wheels/16/02/8a/3a8b533174e4f7691a8fd72dab4493fb6819b79f8fcc1d18a6
Successfully built micromlgen


### Step 2: Prepare Your SisFall Data

Create a directory named `SisFall_Data` in the same location as this notebook (or specify the full path below). Copy all the `.txt` files from your SisFall dataset into this directory.

### Step 3: Run the AI Training Script

The following script will:
1.  **Read and Process Data**: It iterates through the `.txt` files in `SisFall_Data`.
    *   It filters out the last 3 columns (assuming they are redundant).
    *   It downsamples the data from 200Hz to 5Hz by averaging every 40 rows.
    *   It creates overlapping windows of 25 samples (5 seconds) with a step of 12 for robust training.
    *   Labels are assigned based on the filename prefix ('F' for fall, 'D' for daily activities).
2.  **Train SVM Model**: A linear Support Vector Machine (SVM) is trained on the processed data.
3.  **Export C Code**: The trained SVM model is then exported as a C header file (`model.h`) which can be used on STM32 microcontrollers.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_curve

# 1. ĐƯỜNG DẪN DỮ LIỆU SISFALL
DATA_DIR = './Fall Detection csv'

X = []
y = []
groups = []

print("⏳ Đang xử lý Data và Đồng bộ Trục (Axis Remapping)...")

for root, _, files in os.walk(DATA_DIR):
    for filename in files:
        if not filename.endswith('.csv'):
            continue

        label = 1 if filename.startswith('F') else 0
        filepath = os.path.join(root, filename)

        try:
            df = pd.read_csv(filepath, usecols=[0,1,2,3,4,5])
        except Exception as e:
            continue

        # Đọc dữ liệu gốc
        df.columns = ['raw_x', 'raw_y', 'raw_z', 'gx', 'gy', 'gz']

        # 🔴 ĐỒNG BỘ TRỤC: Gán trục dọc của SisFall (Y) vào biến Ax của mạch thật
        df['ax'] = df['raw_y']
        df['ay'] = df['raw_x']
        df['az'] = df['raw_z']

        # SCALE ĐƠN VỊ
        df['ax'] = df['ax'] / 256.0
        df['ay'] = df['ay'] / 256.0
        df['az'] = df['az'] / 256.0
        df['gx'] = df['gx'] / 14.375
        df['gy'] = df['gy'] / 14.375
        df['gz'] = df['gz'] / 14.375

        # DOWNSAMPLE (200Hz -> 5Hz)
        df_downsampled = df.groupby(np.arange(len(df)) // 40).mean()

        # Tính SVM tổng hợp để tìm đỉnh va chạm
        acc_svm_full = np.sqrt(df_downsampled['ax']**2 + df_downsampled['ay']**2 + df_downsampled['az']**2)

        # 🟡 FIX SUY KIỆT DỮ LIỆU & MÔ PHỎNG REAL-TIME BẰNG OFFSET
        windows_to_process = []

        if label == 1:
            peak_idx = acc_svm_full.idxmax()
            # Cắt 3 window lệch nhau (Offset -5, 0, 5) để AI quen với đỉnh va chạm trượt
            for offset in [-5, 0, 5]:
                start_idx = max(0, peak_idx - 5 + offset)
                end_idx = start_idx + 25
                if end_idx <= len(df_downsampled) and len(df_downsampled.iloc[start_idx:end_idx]) == 25:
                    windows_to_process.append(df_downsampled.iloc[start_idx:end_idx])
        else:
            # Nếu là file sinh hoạt bình thường (ADL)
            for i in range(0, len(df_downsampled) - 25, 12):
                window = df_downsampled.iloc[i : i+25]
                if len(window) == 25:
                    windows_to_process.append(window)

        # TRÍCH XUẤT 9 ĐẶC TRƯNG TỪ CÁC WINDOW ĐÃ CHỌN
        for window in windows_to_process:
            acc_svm = np.sqrt(window['ax']**2 + window['ay']**2 + window['az']**2)
            gyro_svm = np.sqrt(window['gx']**2 + window['gy']**2 + window['gz']**2)

            # Tính Tilt Angle (Ax là trục dọc)
            y_val = np.sqrt(window['ay']**2 + window['az']**2)
            x_val = window['ax'] + 1e-6
            tilt_angles = np.arctan2(y_val, x_val) * (180.0 / np.pi)

            f_acc_max = acc_svm.max()
            f_acc_min = acc_svm.min()
            f_acc_std = acc_svm.std()
            f_gyro_max = gyro_svm.max()
            f_gyro_std = gyro_svm.std()
            f_ax_std = window['ax'].std()
            f_ay_std = window['ay'].std()
            f_az_std = window['az'].std()
            f_tilt_mean = tilt_angles.mean()

            features = [
                f_acc_max, f_acc_min, f_acc_std,
                f_gyro_max, f_gyro_std,
                f_ax_std, f_ay_std, f_az_std,
                f_tilt_mean
            ]

            X.append(features)
            y.append(label)
            groups.append(filename)

X = np.array(X)
y = np.array(y)
groups = np.array(groups)

print(f"Tổng số mẫu chuẩn (Không nhiễu): {len(X)} (Té ngã: {sum(y == 1)}, Bình thường: {sum(y == 0)})")

# CHIA TẬP TRAIN & TEST
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# CHUẨN HÓA DỮ LIỆU
print("⚖️ Đang chuẩn hóa dữ liệu...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# HUẤN LUYỆN MODEL (Bỏ probability=True)
print("🧠 Đang huấn luyện SVM tuyến tính...")
clf = SVC(kernel='linear', C=1.0, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

# 🟡 THRESHOLD TUNING
y_scores = clf.decision_function(X_train_scaled)
precisions, recalls, thresholds = precision_recall_curve(y_train, y_scores)
target_recall = 0.95
idx = np.where(recalls >= target_recall)[0][-1]
optimal_threshold = thresholds[idx]

print(f"\n🎯 Đã tìm thấy Ngưỡng an toàn (Threshold): {optimal_threshold:.4f}")

# KIỂM TRA TRÊN TẬP TEST VỚI NGƯỠNG MỚI
y_test_scores = clf.decision_function(X_test_scaled)
y_pred_tuned = (y_test_scores > optimal_threshold).astype(int)

print("\nBÁO CÁO KẾT QUẢ TRÊN TẬP TEST:")
print(classification_report(y_test, y_pred_tuned, target_names=['Binh thuong (0)', 'Te nga (1)']))

# ==========================================================
# XUẤT CODE C HOÀN CHỈNH
# ==========================================================
print("\n// =========================================================")
print("// COPY TOÀN BỘ ĐOẠN DƯỚI ĐÂY DÁN ĐÈ VÀO FILE main.c CỦA BẠN")
print("// =========================================================")

scaler_mean = scaler.mean_
scaler_scale = scaler.scale_
svm_weights = clf.coef_[0]
svm_bias = clf.intercept_[0]

print(f"static const float SVM_BIAS = {svm_bias:.6f}f;")
print(f"static const float SVM_THRESHOLD = {optimal_threshold:.6f}f;\n")

print("static const float SVM_SCALE_MEAN[9] = {", end="")
print(", ".join([f"{x:.6f}f" for x in scaler_mean]), end="};\n")

print("static const float SVM_SCALE_STD[9] = {", end="")
print(", ".join([f"{x:.6f}f" for x in scaler_scale]), end="};\n")

print("static const float SVM_COEF[9] = {", end="")
print(", ".join([f"{x:.6f}f" for x in svm_weights]), end="};\n\n")

print("""uint8_t AI_Predict_Fall(float* features) {
    float score = SVM_BIAS;
    for(int i = 0; i < 9; i++) {
        float scaled_feature = (features[i] - SVM_SCALE_MEAN[i]) / SVM_SCALE_STD[i];
        score += scaled_feature * SVM_COEF[i];
    }
    return (score > SVM_THRESHOLD) ? 1 : 0;
}""")
print("// =========================================================")

⏳ Đang xử lý Data và Đồng bộ Trục (Axis Remapping)...
Tổng số mẫu chuẩn (Không nhiễu): 21566 (Té ngã: 5173, Bình thường: 16393)
⚖️ Đang chuẩn hóa dữ liệu...
🧠 Đang huấn luyện SVM tuyến tính...

🎯 Đã tìm thấy Ngưỡng an toàn (Threshold): 1.1021

BÁO CÁO KẾT QUẢ TRÊN TẬP TEST:
                 precision    recall  f1-score   support

Binh thuong (0)       0.99      0.99      0.99      3419
     Te nga (1)       0.97      0.96      0.97       960

       accuracy                           0.99      4379
      macro avg       0.98      0.98      0.98      4379
   weighted avg       0.99      0.99      0.99      4379


// =========================================================
// COPY TOÀN BỘ ĐOẠN DƯỚI ĐÂY DÁN ĐÈ VÀO FILE main.c CỦA BẠN
// =========================================================
static const float SVM_BIAS = -2.294078f;
static const float SVM_THRESHOLD = 1.102146f;

static const float SVM_SCALE_MEAN[9] = {1.456695f, 0.716799f, 0.177029f, 113.295808f, 28.059260f, 0.245008f

In [ ]:
!unzip Falldetectioncsv.zip

Archive:  Falldetectioncsv.zip
   creating: Fall Detection csv/
  inflating: Fall Detection csv/.DS_Store  
   creating: Fall Detection csv/SA01/
  inflating: Fall Detection csv/SA01/D01_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D02_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D03_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D04_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R02.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R03.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R04.csv  
  inflating: Fall Detection csv/SA01/D05_SA01_R05.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R01.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R02.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R03.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R04.csv  
  inflating: Fall Detection csv/SA01/D06_SA01_R05.csv  
  inflating: Fall Detection csv/SA01/D07_SA01_R01.csv  
  inflating: F